# Regularization

## Learning Objectives
1. Implement L1 and L2 penalties from scratch and observe L1's sparsity effect
2. Compare L1, L2, ElasticNet, and Dropout in PyTorch on the same task
3. Understand why AdamW ≠ Adam + L2 and when AdamW matters
4. Select optimal regularization strength via train/val loss sweep


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import copy, matplotlib
matplotlib.use("Agg")   # non-interactive backend for script mode
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1 — L1 and L2 Penalties from Scratch

In [ ]:
# ── Level 1: manual L1/L2 regularisation (numpy) ─────────────────────────
import numpy as np

np.random.seed(42)
X = np.random.randn(100, 10).astype(np.float32)
# True weights: only 3 of 10 features matter
true_w = np.array([2.0, -1.5, 0.8, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
y = X @ true_w + 0.1 * np.random.randn(100)

# Gradient descent with manual L1 or L2 penalty
def gd_with_reg(X, y, lambda_=0.1, reg="l2", lr=0.01, epochs=300):
    w = np.zeros(X.shape[1])
    for _ in range(epochs):
        y_hat  = X @ w
        grad   = -2 * X.T @ (y - y_hat) / len(y)
        if reg == "l2":
            grad += 2 * lambda_ * w
        elif reg == "l1":
            grad += lambda_ * np.sign(w)
        w -= lr * grad
    return w

w_l2 = gd_with_reg(X, y, lambda_=0.1, reg="l2")
w_l1 = gd_with_reg(X, y, lambda_=0.1, reg="l1")

print("True weights:  ", np.round(true_w, 3))
print("L2 recovered:  ", np.round(w_l2, 3))
print("L1 recovered:  ", np.round(w_l1, 3))

nonzero_l2 = (np.abs(w_l2) > 1e-3).sum()
nonzero_l1 = (np.abs(w_l1) > 1e-3).sum()
print(f"\nL2 non-zero weights: {nonzero_l2}/10  (L2 shrinks but rarely zeros)")
print(f"L1 non-zero weights: {nonzero_l1}/10  (L1 induces sparsity)")


## Level 2 — L1, L2, ElasticNet, Dropout Comparison in PyTorch

In [ ]:
# ── Level 2: compare regularisation strategies ────────────────────────────
import torch, copy
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)

# Deliberately overfit-prone: 200 samples, 100 features
X = torch.randn(200, 100)
w_true = torch.zeros(100)
w_true[:5] = torch.tensor([3., -2., 1.5, -1., 0.5])
y = (X @ w_true + 0.3 * torch.randn(200) > 0).long()

train_ld = DataLoader(TensorDataset(X[:150], y[:150]), batch_size=32, shuffle=True)
X_val, y_val = X[150:].to(device), y[150:].to(device)

def make_model():
    return nn.Sequential(nn.Linear(100, 64), nn.ReLU(),
                         nn.Linear(64, 2)).to(device)

def l1_penalty(model, lambda_):
    return lambda_ * sum(p.abs().sum() for p in model.parameters())

def train_model(model, reg_fn=None, dropout=False, epochs=60, lr=1e-3):
    model = copy.deepcopy(model)
    if dropout:
        # insert dropout after first ReLU
        model = nn.Sequential(model[0], model[1],
                               nn.Dropout(0.5), model[2]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0.0)
    for _ in range(epochs):
        model.train()
        for xb, yb in train_ld:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            if reg_fn:
                loss = loss + reg_fn(model)
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        acc = (model(X_val).argmax(1) == y_val).float().mean().item()
    return acc

base_model = make_model()
configs = {
    "No reg":    (base_model, None,                                 False),
    "L2 (wd)":   (make_model(), None,                               False),  # via weight_decay
    "L1 (lam=0.001)": (make_model(), lambda m: l1_penalty(m, 1e-3), False),
    "Dropout":   (make_model(), None,                               True),
}

# L2 via weight_decay needs its own training loop
def train_l2_wd(epochs=60, wd=1e-3):
    m = make_model()
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=wd)
    for _ in range(epochs):
        m.train()
        for xb, yb in train_ld:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            F.cross_entropy(m(xb), yb).backward()
            opt.step()
    m.eval()
    with torch.no_grad():
        return (m(X_val).argmax(1) == y_val).float().mean().item()

print(f"{'Config':<22}  Val Acc")
print("-" * 34)
for name, (m, reg_fn, dp) in configs.items():
    if name == "L2 (wd)":
        acc = train_l2_wd()
    else:
        acc = train_model(m, reg_fn, dp)
    print(f"{name:<22}  {acc:.3f}")


## Real-World Example 1 — AdamW vs Adam + L2

In [ ]:
# ── RW1: AdamW ≠ Adam + L2  (weight decay decoupling) ────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import copy

torch.manual_seed(42)

# Why it matters: Adam applies adaptive scaling to the regularisation gradient,
# making L2 in Adam effectively smaller for large-gradient params — bad for LLM
# fine-tuning. AdamW decouples weight decay from the gradient update.

X = torch.randn(600, 32)
y = (X[:, 0] + X[:, 2] > 0).long()
loader = DataLoader(TensorDataset(X[:500], y[:500]), batch_size=32, shuffle=True)
X_v, y_v = X[500:].to(device), y[500:].to(device)

def make_net():
    return nn.Sequential(nn.Linear(32, 128), nn.ReLU(),
                         nn.Linear(128, 64),  nn.ReLU(),
                         nn.Linear(64, 2)).to(device)

def train_opt(opt_cls, wd, epochs=30):
    net = make_net()
    opt = opt_cls(net.parameters(), lr=1e-3, weight_decay=wd)
    for _ in range(epochs):
        net.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            F.cross_entropy(net(xb), yb).backward()
            opt.step()
    net.eval()
    with torch.no_grad():
        acc = (net(X_v).argmax(1) == y_v).float().mean().item()
    # L2-norm of all weights (lower = more regularised)
    l2 = sum(p.pow(2).sum().item() for p in net.parameters()) ** 0.5
    return acc, l2

acc_adam, l2_adam = train_opt(torch.optim.Adam,  wd=1e-2)
acc_admw, l2_admw = train_opt(torch.optim.AdamW, wd=1e-2)

print("Optimizer comparison (same weight_decay=0.01):")
print(f"  Adam  — val_acc={acc_adam:.3f}  weight L2-norm={l2_adam:.2f}")
print(f"  AdamW — val_acc={acc_admw:.3f}  weight L2-norm={l2_admw:.2f}")
print("\nAdamW often yields lower weight norms (better decoupled regularisation).")
print("For fine-tuning large models, use AdamW — it is the standard in Hugging Face.")


## Real-World Example 2 — L1 for Sparse Feature Selection

In [ ]:
# ── RW2: L1 regularised logistic regression for feature selection ─────────
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

np.random.seed(42)

# Synthetic data: 50 features, only 5 are informative
n_samples, n_features, n_informative = 500, 50, 5
X = np.random.randn(n_samples, n_features)
true_coef = np.zeros(n_features)
true_coef[:n_informative] = [3., -2., 1.5, -1., 0.8]
y = (X @ true_coef + 0.2 * np.random.randn(n_samples) > 0).astype(int)

X_train, X_test = X[:400], X[400:]
y_train, y_test = y[:400], y[400:]

# L1 (Lasso-style) logistic regression
pipe_l1 = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(penalty="l1", C=0.5, solver="liblinear",
                                  max_iter=500, random_state=42))
])
pipe_l1.fit(X_train, y_train)
coef_l1 = pipe_l1.named_steps["clf"].coef_[0]
nonzero = (np.abs(coef_l1) > 1e-4).sum()

# L2 for comparison
pipe_l2 = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(penalty="l2", C=0.5, solver="lbfgs",
                                  max_iter=500, random_state=42))
])
pipe_l2.fit(X_train, y_train)
coef_l2 = pipe_l2.named_steps["clf"].coef_[0]

acc_l1 = pipe_l1.score(X_test, y_test)
acc_l2 = pipe_l2.score(X_test, y_test)

print(f"L1 — test acc={acc_l1:.3f}  non-zero coefficients={nonzero}/{n_features}")
print(f"L2 — test acc={acc_l2:.3f}  non-zero coefficients={(np.abs(coef_l2)>1e-4).sum()}/{n_features}")

print("\nTop-10 absolute L1 coefficients (should align with features 0-4):")
top10 = np.argsort(np.abs(coef_l1))[::-1][:10]
for rank, idx in enumerate(top10):
    print(f"  Feature {idx:02d}: coef={coef_l1[idx]:+.4f}  "
          f"{'<-- informative' if idx < n_informative else ''}")


## Real-World Example 3 — Regularization Strength Sweep

In [ ]:
# ── RW3: lambda sweep — find optimal regularisation ──────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np, copy

torch.manual_seed(42)

X = torch.randn(400, 20)
y = (X[:, 0] - X[:, 1] > 0).long()
loader_tr = DataLoader(TensorDataset(X[:300], y[:300]), batch_size=32, shuffle=True)
X_tr, y_tr = X[:300].to(device), y[:300].to(device)
X_val, y_val = X[300:].to(device), y[300:].to(device)

def make_net():
    return nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 2)).to(device)

def train_sweep(reg="l2", lam=0.0, epochs=40):
    net = make_net()
    wd  = lam if reg == "l2" else 0.0
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=wd)
    for _ in range(epochs):
        net.train()
        for xb, yb in loader_tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(net(xb), yb)
            if reg == "l1":
                loss = loss + lam * sum(p.abs().sum() for p in net.parameters())
            loss.backward()
            opt.step()
    net.eval()
    with torch.no_grad():
        tr_loss  = F.cross_entropy(net(X_tr), y_tr).item()
        val_loss = F.cross_entropy(net(X_val), y_val).item()
        val_acc  = (net(X_val).argmax(1) == y_val).float().mean().item()
    return tr_loss, val_loss, val_acc

lambdas = [0.0, 1e-4, 1e-3, 5e-3, 1e-2, 5e-2, 0.1]
print(f"{'Lambda':>10}  {'Reg':>5}  {'Train Loss':>11}  {'Val Loss':>10}  {'Val Acc':>8}")
print("-" * 55)
for lam in lambdas:
    for reg in ["l2", "l1"]:
        tr, vl, va = train_sweep(reg=reg, lam=lam)
        print(f"{lam:>10.4f}  {reg:>5}  {tr:>11.4f}  {vl:>10.4f}  {va:>8.3f}")
print("\nLook for lambda where val_loss is minimised without large train_loss gap.")
